# ADAM Simulations - NonConvex

## Parameters

In [0]:
from Solver import NonlocalSolverMomentumAdam, AdamMomentum
from sklearn.model_selection import ParameterGrid
import jax 
import jax.numpy as jnp
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
import numpy as np

param_grid = {'lr': [0.1, 0.01], 'beta1': [0.0, 0.9],'beta2': [0.99, 0.999]}
n_learning_rates = len(param_grid['lr'])
param_list = list(ParameterGrid(param_grid))

dL = lambda y: y*(y**2 -1)
f = lambda x, y: 0.0

# Crear carpeta para guardar figuras si no existe
figures_dir = "figures"
os.makedirs(figures_dir, exist_ok=True)

## Adam - Discrete

In [0]:
# Adam - Discrete (JAX) | Guardar PNG por condición inicial separada
inits = [0.1, -0.1]

for theta_initial in inits:
    # Crear figuras nuevas para ESTA condición inicial
    fig_theta = make_subplots(
        rows=1, cols=n_learning_rates, 
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )
    fig_m = make_subplots(
        rows=1, cols=n_learning_rates, 
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )
    fig_v = make_subplots(
        rows=1, cols=n_learning_rates, 
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )

    for i, lr in enumerate(param_grid['lr']):
        filtered_params = [p for p in param_list if p['lr'] == lr]
        epochs = 100 if lr == 0.1 else 200

        for params in filtered_params:
            print(f'\nAdam Configuration: {params}, theta_initial={theta_initial}')
            solver = AdamMomentum(
                dL=dL, lr=lr, beta1=params['beta1'], beta2=params['beta2'], epochs=epochs
            )
            solver.solve(theta_initial=theta_initial)

            label = f"θ₀={theta_initial}, β₁={params['beta1']}, β₂={params['beta2']}"

            # Theta
            fig_theta.add_trace(go.Scatter(
                x=list(range(epochs)),
                y=solver.theta_result,
                mode='lines',
                name=label,
                legendgroup=f'LR={lr}',
            ), row=1, col=i+1)

            # m
            fig_m.add_trace(go.Scatter(
                x=list(range(epochs)),
                y=solver.m_result,
                mode='markers',
                marker=dict(size=3),
                name=label,
                legendgroup=f'LR={lr}',
            ), row=1, col=i+1)

            # v
            fig_v.add_trace(go.Scatter(
                x=list(range(epochs)),
                y=solver.v_result,
                mode='markers',
                marker=dict(size=3),
                name=label,
                legendgroup=f'LR={lr}',
            ), row=1, col=i+1)

    # Layouts para ESTA condición inicial
    fig_theta.update_layout(
        title_text=f'Theta convergence (Adam) — θ₀={theta_initial}', showlegend=True, width=1500, height=600
    )
    fig_theta.update_xaxes(title_text="k")
    fig_theta.update_yaxes(tickformat=".1f", title_text="Theta_k")

    fig_m.update_layout(
        title_text=f'First moment m (Adam) — θ₀={theta_initial}', showlegend=True, width=1500, height=600
    )
    fig_m.update_xaxes(title_text="k")
    fig_m.update_yaxes(title_text="m_k")

    fig_v.update_layout(
        title_text=f'Second moment v (Adam) — θ₀={theta_initial}', showlegend=True, width=1500, height=600
    )
    fig_v.update_xaxes(title_text="k")
    fig_v.update_yaxes(tickformat=".1f", title_text="v_k")

    # Sufijo de nombre de archivo seguro (sin puntos)
    sign = "pos" if theta_initial > 0 else "neg"
    val = str(abs(theta_initial)).replace('.', 'p')  # 0.1 -> 0p1
    suffix = f"theta0_{sign}{val}"

    # Guardar PNGs en carpeta figures
    fig_theta.write_image(os.path.join(figures_dir, f"adam_theta_ncvx_{suffix}.png"))
    fig_m.write_image(os.path.join(figures_dir, f"adam_m_ncvx_{suffix}.png"))
    fig_v.write_image(os.path.join(figures_dir, f"adam_v_ncvx_{suffix}.png"))

    print(f"Figuras guardadas para θ₀={theta_initial} en '{figures_dir}'")

## Nonlocal Adam

In [0]:
# --- NoLocal Continuous Adam | Guardar PNG por condición inicial separada ---

inits = [0.1, -0.1]

for theta_initial in inits:
    # 1) Crear figuras nuevas para ESTA condición inicial
    fig_theta = make_subplots(
        rows=1, cols=n_learning_rates,
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )
    fig_m = make_subplots(
        rows=1, cols=n_learning_rates,
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )
    fig_v = make_subplots(
        rows=1, cols=n_learning_rates,
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )

    # Títulos específicos por condición inicial
    fig_theta.update_layout(
        title_text=f'Theta values convergence — Nonlocal Continuous Adam (θ₀={theta_initial})'
    )
    fig_m.update_layout(
        title_text=f'First moment (m) — Nonlocal Continuous Adam (θ₀={theta_initial})'
    )
    fig_v.update_layout(
        title_text=f'Second moment (v) — Nonlocal Continuous Adam (θ₀={theta_initial})'
    )

    # 2) Bucle por LR y parámetros
    for i, lr in enumerate(param_grid['lr']):
        if lr == 0.1:
            epochs = 100
        elif lr == 0.01:
            epochs = 200

        t = [0.0, epochs * lr]

        filtered_params = [p for p in param_list if p['lr'] == lr]
        for params in filtered_params:
            print(f'\nNonlocal Continuous Adam Configuration: {params}, theta_initial={theta_initial}')

            # Inicializa con la condición que toca en este ciclo
            solver = NonlocalSolverMomentumAdam(
                f=f, dL=dL, t_span=t, y0=jnp.array([theta_initial]),
                alpha=params['lr'], betas=[params['beta1'], params['beta2']]
            )
            t_values, y_values = solver.solve()

            label = f"β₁={params['beta1']}, β₂={params['beta2']}"

            # Theta
            fig_theta.add_trace(go.Scatter(
                x=t_values / params['lr'],          # eje x normalizado
                y=np.asarray(y_values).squeeze(),   # θ(t)
                mode='lines',
                name=label,
                legendgroup=f'LR={lr}',
            ), row=1, col=i+1)

            # m y v (guardados en el solver)
            numerators   = np.asarray(solver._last_m)  # columnas: [t, m]
            denominators = np.asarray(solver._last_v)  # columnas: [t, v]

            fig_m.add_trace(go.Scatter(
                x=numerators[:, 0] / params['lr'],
                y=numerators[:, 1],
                mode='markers',
                marker=dict(size=3),
                name=label,
                legendgroup=f'LR={lr}',
            ), row=1, col=i+1)

            fig_v.add_trace(go.Scatter(
                x=denominators[:, 0] / params['lr'],
                y=denominators[:, 1],
                mode='markers',
                marker=dict(size=3),
                name=label,
                legendgroup=f'LR={lr}',
            ), row=1, col=i+1)

    # 3) Ejes y tamaños
    fig_theta.update_xaxes(title_text="t/alpha")
    fig_theta.update_yaxes(tickformat=".1f", title_text="Theta(t)")
    fig_theta.update_layout(width=1500, height=600)

    fig_m.update_xaxes(title_text="t/alpha")
    fig_m.update_yaxes(title_text="m(t)")
    fig_m.update_layout(width=1500, height=600)

    fig_v.update_xaxes(title_text="t/alpha")
    fig_v.update_yaxes(title_text="v(t)")
    fig_v.update_layout(width=1500, height=600)

    # 4) Sufijo de nombre de archivo (sin puntos)
    sign = "pos" if theta_initial > 0 else "neg"
    val = str(abs(theta_initial)).replace('.', 'p')  # 0.1 -> 0p1
    suffix = f"theta0_{sign}{val}"

    # 5) Guardar las figuras para ESTA condición inicial
    fig_theta.write_image(os.path.join(figures_dir, f"nonlocal_adam_theta_ncvx_{suffix}.png"))
    fig_m.write_image(os.path.join(figures_dir, f"nonlocal_adam_m_ncvx_{suffix}.png"))
    fig_v.write_image(os.path.join(figures_dir, f"nonlocal_adam_v_ncvx_{suffix}.png"))

    print(f"Figuras guardadas para θ₀={theta_initial} en la carpeta '{figures_dir}'")


## Both Models Together

In [0]:
# --- Adam (discreto) vs. Adam continuo no local
# --- PNGs separados por condición inicial θ0 ---

config_colors = {
    (0.9, 0.99): 'blue',
    (0.9, 0.999): 'green',
    (0.0, 0.99): 'red',
    (0.0, 0.999): 'purple'
}

inits = [0.1, -0.1]

for theta_initial in inits:
    # Crear figuras para ESTA condición inicial
    fig_theta = make_subplots(
        rows=1, cols=2,
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )
    fig_m = make_subplots(
        rows=1, cols=2,
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )
    fig_v = make_subplots(
        rows=1, cols=2,
        subplot_titles=[f'Learning Rate = {lr}' for lr in param_grid['lr']]
    )

    # Iterar por cada learning rate
    for i, lr in enumerate(param_grid['lr']):

        # Filtrar parámetros por LR
        filtered_params = [p for p in param_list if p['lr'] == lr]

        # Epochs y t-span (como en tu código)
        if lr == 0.1:
            epochs = 100
            t_inicial = lr
        elif lr == 0.01:
            epochs = 200
            t_inicial = lr

        t = [t_inicial, epochs * lr]

        # -------- Adam discreto --------
        for params in filtered_params:
            print(f'\nAdam Configuration: {params}, theta_initial={theta_initial}')

            solver = AdamMomentum(
                dL=dL, lr=lr, beta1=params['beta1'], beta2=params['beta2'], epochs=epochs
            )
            solver.solve(theta_initial=theta_initial)

            color = config_colors[(params['beta1'], params['beta2'])]

            # θ_k
            fig_theta.add_trace(go.Scatter(
                x=list(range(epochs)),
                y=solver.theta_result,
                mode='markers',
                marker=dict(symbol='x', size=4, line=dict(width=0.01), color=color),
                name=f'Adam beta1={params["beta1"]}, beta2={params["beta2"]}',
                legendgroup=f'Adam {params["beta1"]},{params["beta2"]}',
                showlegend=(i == 0)
            ), row=1, col=i+1)

            # m_k
            fig_m.add_trace(go.Scatter(
                x=list(range(epochs)),
                y=solver.m_result,
                mode='markers',
                marker=dict(symbol='x', size=4, line=dict(width=0.01), color=color),
                name=f'Adam beta1={params["beta1"]}, beta2={params["beta2"]}',
                legendgroup=f'Adam {params["beta1"]},{params["beta2"]}',
                showlegend=(i == 0)
            ), row=1, col=i+1)

            # v_k
            fig_v.add_trace(go.Scatter(
                x=list(range(epochs)),
                y=solver.v_result,
                mode='markers',
                marker=dict(symbol='x', size=4, line=dict(width=0.01), color=color),
                name=f'Adam beta1={params["beta1"]}, beta2={params["beta2"]}',
                legendgroup=f'Adam {params["beta1"]},{params["beta2"]}',
                showlegend=(i == 0)
            ), row=1, col=i+1)

        # ---- Adam continuo no local ----
        for params in filtered_params:
            color = config_colors[(params['beta1'], params['beta2'])]
            print(f'\nNonlocal Continuous Adam Configuration: {params}, theta_initial={theta_initial}')

            solver_nonlocal = NonlocalSolverMomentumAdam(
                f=f, dL=dL, t_span=t, y0=jnp.array([theta_initial]),
                alpha=params['lr'], betas=[params['beta1'], params['beta2']]
            )
            t_values, y_values = solver_nonlocal.solve()

            # θ(t)
            fig_theta.add_trace(go.Scatter(
                x=t_values / params['lr'],
                y=y_values,
                mode='lines',
                line=dict(color=color),
                name=f'Nonlocal Adam beta1={params["beta1"]}, beta2={params["beta2"]}',
                legendgroup=f'Nonlocal Adam {params["beta1"]},{params["beta2"]}',
                showlegend=(i == 0)
            ), row=1, col=i+1)

            # m(t) y v(t) del solver continuo
            numerators   = np.asarray(solver_nonlocal._last_m)  # [t, m]
            denominators = np.asarray(solver_nonlocal._last_v)  # [t, v]

            fig_m.add_trace(go.Scatter(
                x=numerators[:, 0] / params['lr'],
                y=numerators[:, 1],
                mode='lines',
                line=dict(color=color),
                name=f'Nonlocal Adam beta1={params["beta1"]}, beta2={params["beta2"]}',
                legendgroup=f'Nonlocal Adam {params["beta1"]},{params["beta2"]}',
                showlegend=(i == 0)
            ), row=1, col=i+1)

            fig_v.add_trace(go.Scatter(
                x=denominators[:, 0] / params['lr'],
                y=denominators[:, 1],
                mode='lines',
                line=dict(color=color),
                name=f'Nonlocal Adam beta1={params["beta1"]}, beta2={params["beta2"]}',
                legendgroup=f'Nonlocal Adam {params["beta1"]},{params["beta2"]}',
                showlegend=(i == 0)
            ), row=1, col=i+1)

    # Etiquetas y formato (para ESTA θ0)
    for fig, ytxt in [
        (fig_theta, "Theta values"),
        (fig_m, "First moment values"),
        (fig_v, "Second moment values")
    ]:
        fig.update_xaxes(title_text="t/α")
        fig.update_yaxes(title_text=ytxt, tickformat=".1f")
        fig.update_layout(width=1500, height=600, showlegend=True)

    fig_theta.update_layout(title_text=f"Theta values convergence trajectories — θ₀={theta_initial}")
    fig_m.update_layout(title_text=f"First moment (m) trajectories — θ₀={theta_initial}")
    fig_v.update_layout(title_text=f"Second moment (v) trajectories — θ₀={theta_initial}")

    # Sufijo de archivo (sin puntos)
    sign = "pos" if theta_initial > 0 else "neg"
    val = str(abs(theta_initial)).replace('.', 'p')  # 0.1 -> 0p1
    suffix = f"theta0_{sign}{val}"

    # Guardar PNGs para ESTA condición inicial
    fig_theta.write_image(os.path.join(figures_dir, f"adam_vs_nonlocal_theta_ncvx_{suffix}.png"))
    fig_m.write_image(os.path.join(figures_dir, f"adam_vs_nonlocal_m_ncvx_{suffix}.png"))
    fig_v.write_image(os.path.join(figures_dir, f"adam_vs_nonlocal_v_ncvx_{suffix}.png"))

    print(f"Figuras guardadas (θ₀={theta_initial}) en la carpeta '{figures_dir}'")